# Alfred-Coder — Fine-tune on YOUR work (free GPU: Colab or Kaggle)

Fine-tunes **IBM Granite 4.1 8B Instruct** (Alfred-Coder's base) on your own (prompt -> good-output) pairs using Unsloth QLoRA, then exports a **GGUF** you load in LM Studio.

**Before running:** set a GPU runtime. Colab: Runtime -> Change runtime type -> GPU (T4). Kaggle: Notebook settings -> Accelerator -> GPU T4.

**Notes:**
- An 8B model fine-tunes comfortably on a FREE T4 (Colab or **Kaggle**, ~30h/week) — no Colab Pro needed.
- Fine-tuning personalizes to *your style*; it does not make the model generally smarter.
- Training runs on the cloud GPU, never on your CPU.
- **On Kaggle:** add `train.jsonl` as a Dataset (or upload to `/kaggle/working`), swap the `files.upload()` cell for reading that path, and grab the GGUF zip from the Kaggle output panel.

In [ ]:
%%capture
!pip install unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "ibm-granite/granite-4.1-8b-instruct"  # Alfred-Coder base (Granite 4.1 8B)
# 8B fits a free T4 comfortably. (Unsloth may also offer a *-bnb-4bit build you can use here.)
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,
    dtype = None,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

In [ ]:
# Colab: uploads a local file. On Kaggle: set fname to your dataset path, e.g.
#   fname = "/kaggle/input/alfred-finetune/train.jsonl"   (skip the upload line)
from datasets import load_dataset
try:
    from google.colab import files
    print("Upload train.jsonl (built by scripts/build-finetune-jsonl.ps1):")
    up = files.upload()
    fname = list(up.keys())[0]
except Exception:
    fname = "/kaggle/working/train.jsonl"  # <-- edit for Kaggle

ds = load_dataset("json", data_files=fname, split="train")

def fmt(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)}

ds = ds.map(fmt)
print("examples:", len(ds))
print(ds[0]["text"][:500])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LEN,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        seed = 42,
        output_dir = "outputs",
    ),
)
trainer.train()

In [ ]:
# Merge LoRA into the base and export a GGUF (Q4_K_M) for LM Studio.
model.save_pretrained_gguf("alfred-coder", tokenizer, quantization_method = "q4_k_m")

In [ ]:
import shutil
shutil.make_archive("alfred-coder-gguf", "zip", "alfred-coder")
# Colab: downloads to your PC. Kaggle: find alfred-coder-gguf.zip in the output panel.
try:
    from google.colab import files
    files.download("alfred-coder-gguf.zip")
except Exception:
    print("On Kaggle: download alfred-coder-gguf.zip from the Output tab.")

## Load into LM Studio
1. Unzip `alfred-coder-gguf.zip`.
2. LM Studio -> My Models -> reveal the models folder; copy the `.gguf` into `alfred-coder/`.
3. Reload LM Studio, select **alfred-coder**, start the local server (`http://localhost:1234/v1`).
4. Alfred's local-coder will call it there (see `docs/local-coder/LM-STUDIO-SETUP.md`).